In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/rtm1_2501010174/notebooks')
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/rtm1_2501010174/notebooks


# Tugas 1 - Akuisisi Data Nyata melalui API
**Mata kuliah:** Akuisisi dan Manajemen Data (BIFP-243) - INSTIKI

**Topik:** Akomodasi pariwisata di Kabupaten Gianyar, Bali
**Sumber:** OpenStreetMap lewat Overpass API (lisensi ODbL, (c) OpenStreetMap contributors)

Alur: konfigurasi -> robots.txt dan ketentuan API -> cari wilayah -> satu query data -> simpan data mentah -> provenance -> verifikasi.

In [3]:
# 1. Konfigurasi
import csv, json, time, hashlib, os
import datetime as dt
from zoneinfo import ZoneInfo
import requests
from urllib import robotparser

NAMA  = "Francesco Nera Alfi Nanggor"
NIM   = "2501010174"
EMAIL = "alfinanggor@gmail.com"

SUMBER        = "osm_overpass"
KABUPATEN     = "Kabupaten Gianyar"   # untuk judul dan dokumentasi
KATA_WILAYAH  = "Gianyar"             # dipakai mencari wilayah di OSM (nama di OSM bisa tanpa 'Kabupaten')
ADMIN_LEVEL   = "5"                   # tingkat kabupaten/kota di Indonesia (OSM)
TOURISM_TYPES = ["hotel", "guest_house", "hostel", "motel", "apartment", "resort", "chalet"]

ENDPOINT   = "https://overpass-api.de/api/interpreter"
STATUS_URL = "https://overpass-api.de/api/status"
ROBOTS_URL = "https://overpass-api.de/robots.txt"

USER_AGENT = f"INSTIKI-BIFP243-Tugas1/1.0 ({NAMA}; NIM {NIM}; {EMAIL})"
HEADERS    = {"User-Agent": USER_AGENT}

TIMEOUT    = 90
DELAY      = 1.0   # jeda minimal sebelum setiap request (detik)
MAX_RETRY  = 4
DELAY_AWAL = 1.0   # backoff 1, 2, 4, 8 detik

RAW_DIR  = "../data/raw"   # jalankan dari folder notebooks/
DOCS_DIR = "../docs"
WITA = ZoneInfo("Asia/Makassar")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)
print("User-Agent:", USER_AGENT)

User-Agent: INSTIKI-BIFP243-Tugas1/1.0 (Francesco Nera Alfi Nanggor; NIM 2501010174; alfinanggor@gmail.com)


## 2. robots.txt dan ketentuan API (bukti kepatuhan)
Isi robots.txt dicetak apa adanya dan disimpan ke provenance. Brief tugas menyebut kepatuhan dibuktikan lewat robots.txt **atau** ketentuan API; untuk Overpass, ketentuan API yang dipakai: satu query saja (tidak berulang), cek slot lewat `/api/status`, jeda dan User-Agent jujur.

In [4]:
robots_evidence = {"url": ROBOTS_URL, "checked_at": dt.datetime.now(WITA).isoformat(timespec="seconds")}
rp = robotparser.RobotFileParser()
try:
    r = requests.get(ROBOTS_URL, headers=HEADERS, timeout=TIMEOUT)
    robots_evidence["http_status"] = r.status_code
    if r.status_code == 200:
        rp.parse(r.text.splitlines())
        robots_evidence["can_fetch"] = rp.can_fetch(USER_AGENT, ENDPOINT)
        robots_evidence["content"] = r.text[:1500]
        robots_evidence["note"] = "robots.txt ditemukan dan diperiksa"
    else:
        robots_evidence["can_fetch"] = True
        robots_evidence["note"] = f"robots.txt tidak tersedia (HTTP {r.status_code}); ketentuan API tetap diikuti"
except requests.RequestException as e:
    robots_evidence["can_fetch"] = None
    robots_evidence["note"] = f"gagal memeriksa robots.txt: {e}"

print(json.dumps(robots_evidence, indent=2, ensure_ascii=False))

if robots_evidence["can_fetch"] is False:
    robots_evidence["keputusan"] = ("RobotFileParser mengembalikan can_fetch=False untuk User-Agent ini (isi robots.txt tersimpan di atas). Akses tetap dilakukan "
        "karena Overpass API adalah API publik resmi yang direkomendasikan pada brief tugas dan diatur oleh "
        "ketentuan penggunaannya sendiri (satu query, cek /api/status, jeda, User-Agent jujur). "
        "Keputusan ini dicatat terbuka di README butir 4.")
    print("\nPERHATIAN:", robots_evidence["keputusan"])

{
  "url": "https://overpass-api.de/robots.txt",
  "checked_at": "2026-11-16T02:00:28+08:00",
  "http_status": 200,
  "can_fetch": false,
  "content": "User-agent: *\nDisallow: /api/\nDisallow: /munin/\nDisallow: /achavi/\nSitemap: https://z.overpass-api.de/api/sitemap\n",
  "note": "robots.txt ditemukan dan diperiksa"
}

PERHATIAN: RobotFileParser mengembalikan can_fetch=False untuk User-Agent ini (isi robots.txt tersimpan di atas). Akses tetap dilakukan karena Overpass API adalah API publik resmi yang direkomendasikan pada brief tugas dan diatur oleh ketentuan penggunaannya sendiri (satu query, cek /api/status, jeda, User-Agent jujur). Keputusan ini dicatat terbuka di README butir 4.


In [5]:
# Ketentuan API Overpass: cek slot yang tersedia sebelum mengirim query
time.sleep(DELAY)
rs = requests.get(STATUS_URL, headers=HEADERS, timeout=TIMEOUT)
print("HTTP", rs.status_code)
print(rs.text)

HTTP 200
Connected as: 2287386094
Current time: 2026-11-15T18:01:08Z
Announced endpoint: lambert.openstreetmap.de/
Rate limit: 2
2 slots available now.
Currently running queries (pid, space limit, time limit, start time):



## 3. Fungsi pemanggil dengan retry dan exponential backoff (pola Modul 06)

In [6]:
def panggil_overpass(query, label):
    """POST ke Overpass. Timeout, status code, jeda, retry (backoff 1,2,4,8 dtk), Retry-After saat 429."""
    jeda = DELAY_AWAL
    for attempt in range(1, MAX_RETRY + 1):
        time.sleep(DELAY)
        try:
            resp = requests.post(ENDPOINT, data={"data": query}, headers=HEADERS, timeout=TIMEOUT)
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"  [{label} percobaan {attempt}] {type(e).__name__}, tunggu {jeda}s")
            time.sleep(jeda); jeda *= 2
            continue
        print(f"  [{label} percobaan {attempt}] HTTP {resp.status_code}")
        if resp.status_code == 200:
            try:
                return resp.json()          # jangan baca JSON sebelum status dicek
            except ValueError:
                raise RuntimeError("Status 200 tetapi isi bukan JSON")
        elif resp.status_code == 429:
            try:
                tunggu = float(resp.headers.get("Retry-After", jeda))
            except ValueError:
                tunggu = jeda
            print(f"  429 rate limit, tunggu {tunggu}s")
            time.sleep(tunggu)
        elif 500 <= resp.status_code < 600:
            print(f"  {resp.status_code} kesalahan server, tunggu {jeda}s")
            time.sleep(jeda)
        else:
            raise RuntimeError(f"Tidak dapat dipulihkan: {resp.status_code} {resp.reason}")
        jeda *= 2
    raise RuntimeError(f"{label}: gagal setelah {MAX_RETRY} percobaan")

## 4. Cari wilayah di OSM
Nama wilayah di OSM tidak selalu persis "Kabupaten Gianyar", sehingga pencarian dengan nama persis bisa mengembalikan 0 elemen. Langkah ini mencari relasi batas administrasi yang namanya memuat "Gianyar", lalu memilih yang `admin_level` kabupaten.

In [7]:
DIAG = f'''
[out:json][timeout:25];
rel["boundary"="administrative"]["name"~"{KATA_WILAYAH}"];
out tags;
'''
hasil_wilayah = panggil_overpass(DIAG, "cari wilayah")
kandidat = hasil_wilayah.get("elements", [])
print("Kandidat wilayah:")
for e in kandidat:
    t = e.get("tags", {})
    print(f"  relasi {e['id']} | {t.get('name')} | admin_level={t.get('admin_level')}")

pilihan = [e for e in kandidat if e.get("tags", {}).get("admin_level") == ADMIN_LEVEL]
if not pilihan:
    raise RuntimeError("Tidak ada relasi dengan admin_level kabupaten; periksa daftar kandidat di atas "
                       "dan ubah KATA_WILAYAH / ADMIN_LEVEL.")
REL_ID   = pilihan[0]["id"]
REL_NAME = pilihan[0]["tags"].get("name")
print("Dipakai:", REL_ID, REL_NAME)

  [cari wilayah percobaan 1] HTTP 502
  502 kesalahan server, tunggu 1.0s
  [cari wilayah percobaan 2] HTTP 502
  502 kesalahan server, tunggu 2.0s
  [cari wilayah percobaan 3] HTTP 200
Kandidat wilayah:
  relasi 14613952 | Gianyar | admin_level=5
  relasi 20447262 | Gianyar | admin_level=6
  relasi 20447677 | Gianyar | admin_level=7
Dipakai: 14613952 Gianyar


## 5. Query data akomodasi (satu query)

In [8]:
types_regex = "|".join(TOURISM_TYPES)
QUERY = f'''
[out:json][timeout:60];
area(id:{3600000000 + REL_ID})->.a;
nwr["tourism"~"^({types_regex})$"](area.a);
out center tags;
'''
print(QUERY)

accessed_at = dt.datetime.now(WITA)
data = panggil_overpass(QUERY, "data akomodasi")
elements = data.get("elements", [])
assert len(elements) > 0, "Query mengembalikan 0 elemen - periksa wilayah dan filter tourism"
print("Jumlah elemen:", len(elements))
print("Waktu akses  :", accessed_at.isoformat(timespec="seconds"))
print("OSM timestamp:", data.get("osm3s", {}).get("timestamp_osm_base"))


[out:json][timeout:60];
area(id:3614613952)->.a;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|resort|chalet)$"](area.a);
out center tags;

  [data akomodasi percobaan 1] HTTP 502
  502 kesalahan server, tunggu 1.0s
  [data akomodasi percobaan 2] HTTP 200
Jumlah elemen: 905
Waktu akses  : 2026-11-16T02:02:01+08:00
OSM timestamp: 2026-11-15T18:01:19Z


## 6. Ratakan ke tabel dan simpan data mentah
Hanya kolom non-pribadi yang diambil. Kolom kontak (phone, email, website, media sosial, operator, dsb.) sengaja tidak diambil.

In [9]:
COLUMNS = ["osm_type", "osm_id", "name", "tourism", "stars", "rooms",
           "addr_street", "addr_city", "addr_postcode", "lat", "lon"]

rows = []
for el in elements:
    t = el.get("tags", {})
    if el["type"] == "node":
        lat, lon = el.get("lat"), el.get("lon")
    else:
        c = el.get("center", {})
        lat, lon = c.get("lat"), c.get("lon")
    rows.append({
        "osm_type": el["type"], "osm_id": el["id"],
        "name": t.get("name", ""), "tourism": t.get("tourism", ""),
        "stars": t.get("stars", ""), "rooms": t.get("rooms", ""),
        "addr_street": t.get("addr:street", ""), "addr_city": t.get("addr:city", ""),
        "addr_postcode": t.get("addr:postcode", ""),
        "lat": lat, "lon": lon,
    })

raw_name = f"{SUMBER}_{accessed_at:%Y%m%d}.csv"
raw_path = os.path.join(RAW_DIR, raw_name)
if os.path.exists(raw_path):
    raise FileExistsError(f"{raw_path} sudah ada. Data mentah tidak boleh ditimpa/diubah.")
with open(raw_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=COLUMNS)
    w.writeheader()
    w.writerows(rows)
print("Tersimpan:", raw_path, "-", len(rows), "record")

Tersimpan: ../data/raw/osm_overpass_20261116.csv - 905 record


## 7. provenance.json (pola Modul 05)

In [10]:
def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

provenance = {
    "files": [{
        "file_name": raw_name,
        "size_bytes": os.path.getsize(raw_path),
        "md5": md5_of(raw_path),
        "record_count": len(rows),
        "source": "OpenStreetMap (Overpass API)",
        "publisher": "OpenStreetMap contributors",
        "url": ENDPOINT,
        "license": "ODbL 1.0 - (c) OpenStreetMap contributors",
        "acquired_at": accessed_at.isoformat(timespec="seconds"),
        "osm_data_timestamp": data.get("osm3s", {}).get("timestamp_osm_base"),
        "acquired_by": {"name": NAMA, "nim": NIM},
        "method": "API (HTTP POST)",
        "region": {"name": REL_NAME, "osm_relation_id": REL_ID},
        "query": QUERY.strip(),
        "request_delay_seconds": DELAY,
        "user_agent": USER_AGENT,
        "robots_check": robots_evidence,
    }]
}
with open(os.path.join(DOCS_DIR, "provenance.json"), "w", encoding="utf-8") as f:
    json.dump(provenance, f, indent=2, ensure_ascii=False)
print(json.dumps(provenance, indent=2, ensure_ascii=False))

{
  "files": [
    {
      "file_name": "osm_overpass_20261116.csv",
      "size_bytes": 69206,
      "md5": "75e764c65ece88385b81ae70476fd932",
      "record_count": 905,
      "source": "OpenStreetMap (Overpass API)",
      "publisher": "OpenStreetMap contributors",
      "url": "https://overpass-api.de/api/interpreter",
      "license": "ODbL 1.0 - (c) OpenStreetMap contributors",
      "acquired_at": "2026-11-16T02:02:01+08:00",
      "osm_data_timestamp": "2026-11-15T18:01:19Z",
      "acquired_by": {
        "name": "Francesco Nera Alfi Nanggor",
        "nim": "2501010174"
      },
      "method": "API (HTTP POST)",
      "region": {
        "name": "Gianyar",
        "osm_relation_id": 14613952
      },
      "query": "[out:json][timeout:60];\narea(id:3614613952)->.a;\nnwr[\"tourism\"~\"^(hotel|guest_house|hostel|motel|apartment|resort|chalet)$\"](area.a);\nout center tags;",
      "request_delay_seconds": 1.0,
      "user_agent": "INSTIKI-BIFP243-Tugas1/1.0 (Francesco Nera Alf

## 8. Verifikasi terhadap target (>= 100 record, >= 5 kolom bermakna)

In [11]:
with open(raw_path, newline="", encoding="utf-8") as f:
    check = list(csv.DictReader(f))
print("Record :", len(check), "(target >= 100) ->", "OK" if len(check) >= 100 else "KURANG")
print("Kolom  :", len(COLUMNS))
print("Kolom terisi:")
for c in COLUMNS:
    filled = sum(1 for r in check if r[c] not in ("", None))
    print(f"  {c:15s} {filled:4d}/{len(check)}")
print("Jenis akomodasi:")
for j in TOURISM_TYPES:
    print(f"  {j:12s}", sum(1 for r in check if r['tourism'] == j))
print("MD5 cocok:", md5_of(raw_path) == provenance["files"][0]["md5"])

Record : 905 (target >= 100) -> OK
Kolom  : 11
Kolom terisi:
  osm_type         905/905
  osm_id           905/905
  name             830/905
  tourism          905/905
  stars             25/905
  rooms             44/905
  addr_street      261/905
  addr_city        102/905
  addr_postcode    151/905
  lat              905/905
  lon              905/905
Jenis akomodasi:
  hotel        422
  guest_house  351
  hostel       73
  motel        3
  apartment    27
  resort       0
  chalet       29
MD5 cocok: True


## 9. Isi otomatis bagian README yang bergantung pada hasil

In [12]:
# Mengisi {{JAM_AKSES}}, {{HASIL_ROBOTS}}, {{JUMLAH_RECORD}} di README.md dari hasil run ini.
readme_path = os.path.join(DOCS_DIR, "..", "README.md")
jam = accessed_at.strftime("%Y-%m-%d %H:%M WITA")
cf = robots_evidence.get("can_fetch")
if cf is False:
    hasil = "`can_fetch = False`. " + robots_evidence["keputusan"]
elif cf is True:
    hasil = "`can_fetch = True` (" + robots_evidence["note"] + ")."
else:
    hasil = "tidak dapat diperiksa (" + robots_evidence["note"] + ")."
isian = {"{{JAM_AKSES}}": jam, "{{HASIL_ROBOTS}}": hasil,
         "{{JUMLAH_RECORD}}": f"{len(rows)} record, {len(COLUMNS)} kolom"}
if os.path.exists(readme_path):
    with open(readme_path, encoding="utf-8") as f:
        teks = f.read()
    for k, v in isian.items():
        teks = teks.replace(k, v)
    with open(readme_path, "w", encoding="utf-8") as f:
        f.write(teks)
    print("README.md terisi:")
else:
    print("README.md tidak ditemukan; salin nilai ini ke README secara manual:")
for k, v in isian.items():
    print(" ", k, "->", v)

README.md terisi:
  {{JAM_AKSES}} -> 2026-11-16 02:02 WITA
  {{HASIL_ROBOTS}} -> `can_fetch = False`. RobotFileParser mengembalikan can_fetch=False untuk User-Agent ini (isi robots.txt tersimpan di atas). Akses tetap dilakukan karena Overpass API adalah API publik resmi yang direkomendasikan pada brief tugas dan diatur oleh ketentuan penggunaannya sendiri (satu query, cek /api/status, jeda, User-Agent jujur). Keputusan ini dicatat terbuka di README butir 4.
  {{JUMLAH_RECORD}} -> 905 record, 11 kolom
